In [0]:
%run ../utils

In [0]:
# Create lead_activites_processed_custom_fields — THE MISSING JOIN,
# materialized: activity custom VALUES resolved against their field
# DEFINITIONS.
#
# One row per (typed activity, populated field):
#   audit + activity PK + type id + TYPE NAME (from custom_activites_processed)
#   + field key + field VALUE (from the parent's custom_fields MAP)
#   + FIELD NAME + FIELD TYPE + required (from custom_activites_processed_fields).
#
# The definition join uses BOTH the type id and the field id — an id-only
# join explodes ~96x (each field id is defined on ~53 activity types; a
# measured id-only join produced 124M rows against 1.29M values).
# LEFT JOIN + stats: unmatched values are KEPT with NULL name/type so
# resolution coverage stays visible in every run.
# CDC: parent-driven delete+append on the child watermark (same as 2_4_02).
import time
from pyspark.sql.functions import col, lit, explode
start = time.time()

print(f"{'='*80}")
print(f"lead_activites_processed + definitions -> lead_activites_processed_custom_fields")
print(f"{'='*80}")

PARENT_TABLE = 'crm_ingestion.silver.lead_activites_processed'
CHILD_TABLE = 'lead_activites_processed_custom_fields'

watermark = get_silver_watermark(CHILD_TABLE)
if watermark:
    print(f"  CDC MODE: reading parent changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Read the silver parent (children can never be newer than the parent)
df_flat = spark.table(PARENT_TABLE)
if watermark:
    df_flat = df_flat.filter(col("bronze_updated_at") > lit(watermark))

n_changed = df_flat.count()
if n_changed == 0:
    print(f"  CDC: No new parent rows since {watermark}")
    tdf = spark.table(f"crm_ingestion.silver.{CHILD_TABLE}")
    print(f"  Existing {CHILD_TABLE}: {tdf.count():,} rows")
else:
    print(f"  Source: {n_changed:,} changed parent rows")
    typed = df_flat.where(col("custom_activity_type_id").isNotNull())

    # Step 2: Values — one row per populated (activity, field)
    extra = ["lead_id"] if "lead_id" in typed.columns else []
    vals = (typed
            .select("bronze_insert_date", "bronze_updated_at",
                    col("id").alias("lead_activites_processed_id"),
                    "custom_activity_type_id", *extra,
                    explode("custom_fields").alias("field_key", "field_value"))
            .where(col("field_value").isNotNull()))
    n_vals = vals.count()
    print(f"  Populated (activity, field) values in batch: {n_vals:,}")

    # Step 3: Definitions — (type, field) pairs, join on BOTH keys
    defs = (spark.table("crm_ingestion.silver.custom_activites_processed_fields")
            .select(col("custom_activites_processed_id").alias("_def_type_id"),
                    col("id").alias("_def_field_id"),
                    col("name").alias("field_name"),
                    col("type").alias("field_type"),
                    "required"))
    n_def_rows = defs.count()
    n_def_pairs = defs.select("_def_type_id", "_def_field_id").distinct().count()
    print(f"  Definitions (raw): {n_def_rows:,} rows, {n_def_pairs:,} distinct (type, field) pairs")
    # Upstream silver state (Nov 10 2026 run log): custom_activites_processed and its
    # fields child carry 31 snapshot versions per record (899 rows / 29 type ids;
    # 7,285 / 235 pairs — identical definitions, differing only bronze_insert_date).
    # Dedupe BOTH sides before joining, or each value matches 31x31 = 961 rows
    # (this notebook's first run measured a 1.24B-row join before the fix).
    defs = defs.dropDuplicates(["_def_type_id", "_def_field_id"])
    print(f"  Definitions (deduped): {defs.count():,} rows — 1:1 join guaranteed")

    # Step 4: Type names (deduped — same 31-versions-per-id silver state as above)
    types = (spark.table("crm_ingestion.silver.custom_activites_processed")
             .select(col("id").alias("_type_id"),
                     col("name").alias("custom_activity_type_name"))
             .dropDuplicates(["_type_id"]))

    resolved = (vals
        .join(defs,
              (vals["custom_activity_type_id"] == defs["_def_type_id"]) &
              (vals["field_key"] == defs["_def_field_id"]),
              "left")
        .drop("_def_type_id", "_def_field_id")
        .join(types, vals["custom_activity_type_id"] == types["_type_id"], "left")
        .drop("_type_id"))

    n_rows = resolved.count()
    n_resolved = resolved.where(col("field_name").isNotNull()).count()
    print(f"  Resolution: {n_resolved:,} of {n_vals:,} values match a (type, field) definition"
          f" ({100.0 * n_resolved / max(n_vals, 1):.1f}%)")
    if n_rows != n_vals:
        print(f"  WARNING: {n_rows:,} output rows vs {n_vals:,} values — duplicate definitions inflated the join")

    # Step 5: Write (full overwrite on first run, delete+append on CDC)
    _CDC_WATERMARK = watermark
    write_silver(resolved, CHILD_TABLE)
    _CDC_WATERMARK = None

    print(f"\n  Completed in {time.time() - start:.1f}s")
print(f"{'='*80}")

In [0]:
# Sample: resolved values with their field names — the thing the raw MAP could never show
from pyspark.sql.functions import col

display(spark.table('crm_ingestion.silver.lead_activites_processed_custom_fields')
        .where(col('field_name').isNotNull())
        .select('lead_activites_processed_id', 'custom_activity_type_name',
                'field_name', 'field_type', 'field_value')
        .limit(50))